In [2]:
from typing import Literal , Dict , Any
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [10]:
load_dotenv()

True

In [4]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain.messages import SystemMessage
from prompt import PROMPT 

# 1. Output Schema
class QueryRoute(BaseModel):
    intent: Literal["EMERGENCY", "FACILITY_LOOKUP", "SYMPTOM_ASSESSMENT"] = Field(
        description="The primary operational path for the patient's query."
    )
    is_vague: bool = Field(
        description="True if the symptom report lacks duration, severity, or specifics needed for safe triage."
    )
    reasoning: str = Field(
        description="A concise 1-sentence medical justification for the routing decision."
    )

# 2. Router Setup
ROUTER_SYSTEM_PROMPT = PROMPT

def route_patient_query(user_query: str) -> QueryRoute:
    llm =   ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0.0)
    structured_router = llm.with_structured_output(QueryRoute)
    
    prompt = ChatPromptTemplate.from_messages([
        SystemMessage(ROUTER_SYSTEM_PROMPT),
        ("human", "Patient Input: {input}")
    ])
    
    chain = prompt | structured_router
    return chain.invoke({"input": user_query})



In [1]:
from langchain_community.document_loaders import PyPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter

# load documents 
loader1 = PyPDFLoader("docs/symptoms.pdf")
loader2 = PyPDFLoader("docs/medical_info.pdf")
pages = loader1.load()
pages2 = loader2.load()
pages.extend(pages2)

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500 , chunk_overlap=50)
chunks = text_splitter.split_documents(pages)

C:\Users\Gaurav Negi\AppData\Local\Temp\ipykernel_4580\2526086658.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
from langchain_community.retrievers import BM25Retriever
import pickle 

retriever = BM25Retriever.from_documents(chunks)
retriever.k = 4
with open("bm25_retriever.pkl" , "wb") as f :
    pickle.dump(retriever , f)

In [6]:
print(len(chunks))

7387


In [8]:

from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
from pinecone import Pinecone , ServerlessSpec
import os 

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "my-pinecone-index"
if index_name not in pc.list_indexes().names():
    print("yes")
    pc.create_index(
        name=index_name,
        dimension=384,  # Matches OpenAI text-embedding-3-small
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

yes


In [16]:
index = pc.Index(index_name)
vectorstore = PineconeVectorStore(index=index, embedding=embeddings)

In [10]:
from langchain_pinecone import PineconeVectorStore
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate
from pinecone import Pinecone , ServerlessSpec
import os

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "my-pinecone-index"

index = pc.Index(index_name)
vector_store = PineconeVectorStore(index=index, embedding=embeddings)

def setup_hybrid_retriever():

    # Dense vector retriever
    vector_retriever = vector_store.as_retriever(
            search_type = "similarity" , 
            search_kwargs = {
                "k" : 4, 
            }
    )

    # Sparse retriever
    with open("bm25_retriever.pkl" , "rb") as f:
        bm25_retriever = pickle.load(f)
    bm25_retriever.k = 4

    # Ensemble
    hybrid_retriever = EnsembleRetriever(
            retrievers=[bm25_retriever, vector_retriever],
            weights=[0.3, 0.7]
        )

    return hybrid_retriever

hybrid_retriever = setup_hybrid_retriever()


In [73]:
# Agentic workflow controller
def workflow_controller(user_query) : 
    decision = route_patient_query(user_query)
    
    if decision.intent == "EMERGENCY":
        return {
            "action" : decision.intent ,
            "message": f"Critical condition detected. Seek immediate emergency medical care. \n\n {decision.reasoning}"
        }

    elif decision.intent == "FACILITY_LOOKUP":
        return {
            "action" : decision.intent , 
            "message" : ""
        }
    else :
        handle_symptom_path(user_query , decision.is_vague , hybrid_retriever)

In [17]:
from pydantic import BaseModel , Field

class Assesment(BaseModel):
    patien_guidance: str = Field(
        description="A concise 3-4 lines clear, non-diagnostic guidance emphasizing doctor consultation."
    )
    reasoning: str = Field(
        description="Detailed structured notes for phc doctor"
    )

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

def handle_symptom_path(user_query: str, is_vague: bool, hybrid_retriever):
    if is_vague: 
        pass 

    retrieved_docs = hybrid_retriever.invoke(user_query)
    # Branch B: Execute Full Hybrid RAG Pipeline
    retrieved_docs = hybrid_retriever.invoke(user_query)
    context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    prompt_template = """
    You are a clinical decision-support assistant for rural health workers in India.
    Analyze the patient query using ONLY the provided WHO symptoms and triage guidelines.
    
    Context Guidelines:
    {context}
    
    Patient Query: {query}
    
    Generate a JSON response with:
    1. "patient_guidance": Clear, non-diagnostic guidance emphasizing doctor consultation.
    2. "doctor_summary_english": Detailed 5-10 lines Structured clinical notes for a PHC doctor.
    """
    llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0.0).with_structured_output(Assesment)
    prompt = PromptTemplate(template=prompt_template, input_variables=["context", "query"])
    
    chain = prompt | llm
    llm_response = chain.invoke({"context": context_text, "query": user_query})

    return llm_response

    

In [21]:
handle_symptom_path("High fever for 3 days, pain behind the eyes, and red rashes appearing on the body." , False , hybrid_retriever)

c:\Users\Gaurav Negi\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Assesment(patien_guidance="Please visit the nearest Primary Health Centre or consult a qualified doctor immediately for a detailed medical checkup. Your symptoms of high fever, pain behind the eyes, and body rashes require prompt clinical assessment and care. Ensure you stay hydrated and well-rested while seeking medical attention. Do not take any unprescribed medications without a doctor's advice.", reasoning='Patient presents with a 3-day history of high fever, retro-orbital pain (eye pain), and red rashes on the body. Based on the provided context, this triad of symptoms strongly aligns with acute infectious viral illness like dengue fever, which typically exhibits sudden high fever lasting 2 to 7 days accompanied by eye pain and characteristic rash. Given the potential risk of biphasic fever, capillary leak, bleeding tendencies, or progression to shock, immediate evaluation by a PHC doctor is necessary. Recommended next steps: Monitor vital signs (BP, pulse, capillary refill time),